In [ ]:
!pip install --quiet --force-reinstall --no-deps "numpy==1.26.4" "scipy==1.10.1" "scikit-learn==1.3.2" "imgaug==0.4.0"

In [ ]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten, Reshape, Conv2D, Conv2DTranspose, MaxPool2D, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import imgaug.augmenters as iaa
from imgaug.augmentables.segmaps import SegmentationMapsOnImage
import cv2
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [ ]:
def build_autoencoder(img_shape, code_size=1024):
    H, W, C = img_shape

    # Encoder
    encoder = tf.keras.Sequential([
        Conv2D(code_size//4, 3, padding="same", activation='relu', input_shape=img_shape),
        MaxPool2D(2, padding="same"),  # 255 -> 128
        Conv2D(code_size//2, 3, padding="same", activation='relu'),
        MaxPool2D(2, padding="same"),  # 128 -> 64
        Conv2D(code_size, 3, padding="same", activation='relu'),
        MaxPool2D(2, padding="same"),  # 64 -> 32
        Flatten(),
        Dense(code_size, activation='relu')
    ])

    # Dimension after flattening for reshaping into the decoder
    pre_flatten_shape = (32, 32, code_size)

    # Decoder
    decoder = tf.keras.Sequential([
        Dense(16*16*128, activation='relu', input_shape=(latent_dim,)),
        Reshape((16,16,128)),
        Conv2DTranspose(128, 3, strides=2, padding='same', activation='relu'),  # 16->32
        Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu'),   # 32->64
        Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu'),   # 64->128
        Conv2DTranspose(3, 3, padding='same', activation='sigmoid')             # remains 128
    ])



    return encoder, decoder

In [4]:
def data_augment():
    return iaa.Sequential([
        iaa.Dropout((0, 0.05)),
        iaa.Affine(rotate=(-30, 30)),
        iaa.Fliplr(0.5),
        iaa.Crop(percent=(0, 0.2), keep_size=True),
        iaa.WithBrightnessChannels(iaa.Add((-50, 50))),
        iaa.Grayscale(alpha=(0.0, 0.5)),
        iaa.GammaContrast((0.5, 2.0), per_channel=True),
        iaa.PiecewiseAffine(scale=(0.01, 0.1)),
    ], random_order=True)


def data_aug_impl_no_label(image_train, n=1):
    da = data_augment()
    for _ in range(n):
        augmented = da(images=image_train.copy())
        image_train = np.append(image_train, augmented, axis=0)
    return image_train


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

IMG_SHAPE = (128, 128, 3)
latent_dim = 512
batch_size = 8
epochs = 300

dataset_path = "/kaggle/input/dataset-crop-glomeruli/patch_imgs.npy"

# ---- Load dataset ----
data = np.load(dataset_path)


# ---- Generators ----
def data_generator(data_array):
    for img in data_array:
        yield img, img

def process_data(image, target):
    return tf.cast(image, tf.float32)/255, tf.cast(target, tf.float32)/255

def load_generator():
    sig = (tf.TensorSpec(shape=IMG_SHAPE, dtype=tf.int32),
           tf.TensorSpec(shape=IMG_SHAPE, dtype=tf.int32))
    
    train_ds = tf.data.Dataset.from_generator(lambda: data_generator(train_data),
                                              output_signature=sig)\
                .batch(batch_size).map(process_data, num_parallel_calls=tf.data.AUTOTUNE)
    
    val_ds = tf.data.Dataset.from_generator(lambda: data_generator(val_data),
                                            output_signature=sig)\
                .batch(batch_size).map(process_data, num_parallel_calls=tf.data.AUTOTUNE)
    
    test_ds = tf.data.Dataset.from_generator(lambda: data_generator(test_data),
                                             output_signature=sig)\
                .batch(batch_size).map(process_data, num_parallel_calls=tf.data.AUTOTUNE)
    
    return train_ds, val_ds, test_ds


In [ ]:
import cv2  

# ---- Resize all images to 128x128 ----
data_resized = np.array([cv2.resize(img, (128, 128)) for img in data])

# ---- Split in train / val / test ----
train_data, temp_data = train_test_split(data_resized, test_size=0.30, random_state=42, shuffle=True)
val_data, test_data = train_test_split(temp_data, test_size=0.50, random_state=42, shuffle=True)


print("Train prima augmentation:", train_data.shape)
train_data = data_aug_impl_no_label(train_data, 2)
print("Train dopo augmentation:", train_data.shape)

# ---- Generators ----
train_ds, val_ds, test_ds = load_generator()  # now load_generator will use dataset already resized

# ---- Build model ----
encoder, decoder = build_autoencoder(IMG_SHAPE, latent_dim)
inp = Input(IMG_SHAPE)
code = encoder(inp)
out = decoder(code)
autoencoder = tf.keras.Model(inputs=inp, outputs=out)
autoencoder.compile(
    optimizer='adamax',
    loss='mse',
    metrics=['mae']
)

# ---- Callbacks ----
callbacks = [
    EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True),
    ModelCheckpoint("autoencoder_best.keras", monitor="val_loss", save_best_only=True)  # modern format
]

# ---- Training ----
history = autoencoder.fit(train_ds, epochs=epochs, validation_data=val_ds, callbacks=callbacks)

autoencoder.save("/kaggle/working/autoencoder_model.keras")

